In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

batch_size = 128
epochs = 10
lr = 1e-3
mc_samples = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)

In [2]:
class CNN(nn.Module):
    def __init__(self, dropout_p=0.3, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(p=dropout_p)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward_features(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.dropout(x)

        x = self.pool(F.relu(self.conv2(x)))
        x = self.dropout(x)

        x = self.pool(F.relu(self.conv3(x)))
        x = self.dropout(x)

        x = x.view(x.size(0), -1)
        h = F.relu(self.fc1(x))
        return h

    def forward(self, x):
        h = self.forward_features(x)
        h = self.dropout(h)
        logits = self.fc2(h)
        return logits

In [3]:
def estimate_react_threshold(model, id_loader, percentile=90):
    model.to(device)
    model.eval()

    activations = []

    with torch.no_grad():
        for x, _ in id_loader:
            x = x.to(device)
            h = model.forward_features(x)
            activations.append(h.cpu().numpy())

    activations = np.concatenate(activations, axis=0)
    c = np.percentile(activations, percentile)

    print(f"ReAct threshold c (p={percentile}): {c:.4f}")
    return c

In [4]:
def get_react_softmax_ood_scores(model, id_loader, ood_loader, c):
    model.to(device)
    model.eval()

    id_scores = []
    ood_scores = []

    def compute_scores(loader, scores_list):
        with torch.no_grad():
            for x, _ in loader:
                x = x.to(device)

                h = model.forward_features(x)
                h_react = torch.clamp(h, max=c)

                logits = model.fc2(h_react)
                probs = F.softmax(logits, dim=1)
                max_probs, _ = probs.max(dim=1)

                scores = 1.0 - max_probs
                scores_list.append(scores.cpu().numpy())

    compute_scores(id_loader, id_scores)
    compute_scores(ood_loader, ood_scores)

    return np.concatenate(id_scores), np.concatenate(ood_scores)

In [5]:
transform_cifar = transforms.Compose([
    transforms.ToTensor()
])

transform_mnist = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.expand(3, -1, -1))
])

train_id = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
test_id = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar)
test_ood = datasets.MNIST(root='./data', train=False, download=True, transform=transform_mnist)

train_id_loader = DataLoader(train_id, batch_size=batch_size, shuffle=True)
test_id_loader = DataLoader(test_id, batch_size=batch_size, shuffle=False)
test_ood_loader = DataLoader(test_ood, batch_size=batch_size, shuffle=False)

In [11]:
def train(model, train_loader, epochs, lr):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_function = nn.CrossEntropyLoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        correct_samples = 0
        total_samples = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            logits = model(x)
            loss = loss_function(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            preds = logits.argmax(dim=1)

            correct_samples += (preds == y).sum().item()
            total_samples += y.size(0)

        print(f'Epoch {epoch}; Train loss {total_loss / len(train_loader)}; Accuracy {correct_samples / total_samples * 100}')

In [7]:
def compute_ood_metrics(id_scores, ood_scores):
    y_true = np.concatenate([
        np.zeros_like(id_scores),
        np.ones_like(ood_scores)
    ])
    scores = np.concatenate([id_scores, ood_scores])

    auroc = roc_auc_score(y_true, scores)
    aupr = average_precision_score(y_true, scores)

    fpr, tpr, _ = roc_curve(y_true, scores)
    target_tpr = 0.95
    idxs = np.where(tpr >= target_tpr)[0]
    if len(idxs) > 0:
        fpr95 = fpr[idxs[0]]
    else:
        fpr95 = 1.0

    print(f'AUROC {auroc}')
    print(f'AUPR {aupr}')
    print(f'FPR@95%TPR {fpr95}')

    return auroc, aupr, fpr95

In [8]:
def get_softmax_ood_scores(model, id_loader, ood_loader):
    model.to(device)
    model.eval()

    id_scores = []
    ood_scores = []

    with torch.no_grad():
        for x, _ in id_loader:
            x = x.to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            max_probs, _ = probs.max(dim=1)
            scores = 1.0 - max_probs

            id_scores.append(scores.cpu().numpy())

    with torch.no_grad():
        for x, _ in ood_loader:
            x = x.to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            max_probs, _ = probs.max(dim=1)
            scores = 1.0 - max_probs

            ood_scores.append(scores.cpu().numpy())

    id_scores = np.concatenate(id_scores)
    ood_scores = np.concatenate(ood_scores)

    return id_scores, ood_scores


def get_mcd_ood_entropy(model, x, T=20):
    model.to(device)
    model.train()

    with torch.no_grad():
        probs_T = []
        for _ in range(T):
            logits = model(x)
            probs = F.softmax(logits, dim=1)

            probs_T.append(probs.unsqueeze(0))

        probs_T = torch.cat(probs_T, dim=0)

    p_mean = probs_T.mean(dim=0)

    eps = 1e-8
    entropy = -torch.sum(p_mean * torch.log(p_mean + eps), dim=1)

    return entropy


def get_mcd_ood_scores(model, id_loader, ood_loader, T=20):
    model.to(device)

    id_scores = []
    ood_scores = []

    for x, _ in id_loader:
        x = x.to(device)
        entropy = get_mcd_ood_entropy(model, x, T=T)

        id_scores.append(entropy.cpu().numpy())

    for x, _ in ood_loader:
        x = x.to(device)
        entropy = get_mcd_ood_entropy(model, x, T=T)

        ood_scores.append(entropy.cpu().numpy())

    id_scores = np.concatenate(id_scores)
    ood_scores = np.concatenate(ood_scores)

    return id_scores, ood_scores

In [12]:
model = CNN(dropout_p=0.3, num_classes=10)
train(model, train_id_loader, epochs=epochs, lr=lr)

softmax_id, softmax_ood = get_softmax_ood_scores(
    model, test_id_loader, test_ood_loader
)
print("=== Softmax ===")
compute_ood_metrics(softmax_id, softmax_ood)

mcd_id, mcd_ood = get_mcd_ood_scores(
    model, test_id_loader, test_ood_loader, T=mc_samples
)
print("=== MC Dropout ===")
compute_ood_metrics(mcd_id, mcd_ood)

c = estimate_react_threshold(model, train_id_loader, percentile=90)

react_id, react_ood = get_react_softmax_ood_scores(
    model, test_id_loader, test_ood_loader, c
)
print("=== ReAct (Softmax + ReAct) ===")
compute_ood_metrics(react_id, react_ood)


Epoch 1; Train loss 1.7143493223068353; Accuracy 36.866
Epoch 2; Train loss 1.3912502618701867; Accuracy 49.71
Epoch 3; Train loss 1.243469724569784; Accuracy 55.138
Epoch 4; Train loss 1.1426173741250392; Accuracy 59.208000000000006
Epoch 5; Train loss 1.0768145092612946; Accuracy 61.814
Epoch 6; Train loss 1.0098315599324452; Accuracy 64.132
Epoch 7; Train loss 0.9546654946968683; Accuracy 66.258
Epoch 8; Train loss 0.9208818416461311; Accuracy 67.47800000000001
Epoch 9; Train loss 0.8796511268066933; Accuracy 69.106
Epoch 10; Train loss 0.8543550309622684; Accuracy 69.608
=== Softmax ===
AUROC 0.6410576750000001
AUPR 0.5776867872606146
FPR@95%TPR 0.7641
=== MC Dropout ===
AUROC 0.710674705
AUPR 0.6134122775507135
FPR@95%TPR 0.6426
ReAct threshold c (p=90): 1.0614
=== ReAct (Softmax + ReAct) ===
AUROC 0.85582342
AUPR 0.8136316162475459
FPR@95%TPR 0.4268


(np.float64(0.85582342), np.float64(0.8136316162475459), np.float64(0.4268))